Cell 1：Clone + 路径初始化

In [1]:
# ============================================================
# Cell 1
# Fresh Kaggle setup for E18.5 ablation
# ============================================================

from pathlib import Path
import os
import subprocess


REPO_ROOT = Path(
    "/kaggle/working/SpaMGCL"
)

PROJECT_ROOT = (
    REPO_ROOT
    / "SpaMGCL"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


# ------------------------------------------------------------
# Clone
# ------------------------------------------------------------

if not REPO_ROOT.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/huqian122/SpaMGCL.git",
            str(REPO_ROOT),
        ],
        check=True,
    )

else:

    print(
        "Repository already exists:",
        REPO_ROOT,
    )


assert PROJECT_ROOT.exists(), (
    f"Missing project root:\n{PROJECT_ROOT}"
)

assert DATA_ROOT.exists(), (
    f"Missing Kaggle dataset:\n{DATA_ROOT}"
)


os.chdir(
    PROJECT_ROOT
)


print("=" * 100)
print("PATH SETUP")
print("=" * 100)

print(
    "PROJECT_ROOT =",
    PROJECT_ROOT,
)

print(
    "DATA_ROOT    =",
    DATA_ROOT,
)

print(
    "CWD          =",
    Path.cwd(),
)

print(
    "\nPASS: repository and data found."
)

Cloning into '/kaggle/working/SpaMGCL'...


PATH SETUP
PROJECT_ROOT = /kaggle/working/SpaMGCL/SpaMGCL
DATA_ROOT    = /kaggle/input/datasets/wuvdji/smgc-data
CWD          = /kaggle/working/SpaMGCL/SpaMGCL

PASS: repository and data found.


Cell 2：安装/检查环境

In [2]:
# ============================================================
# Cell 2
# Environment setup
# ============================================================

import sys
import subprocess


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "anndata==0.11.4",
        "scikit-learn==1.6.1",
        "PyYAML",
    ],
    check=True,
)


import torch
import numpy as np
import pandas as pd
import sklearn
import anndata
import yaml


print("=" * 100)
print("ENVIRONMENT")
print("=" * 100)

print(
    "Python      :",
    sys.version.split()[0],
)

print(
    "PyTorch     :",
    torch.__version__,
)

print(
    "NumPy       :",
    np.__version__,
)

print(
    "scikit-learn:",
    sklearn.__version__,
)

print(
    "anndata     :",
    anndata.__version__,
)

print(
    "CUDA        :",
    torch.cuda.is_available(),
)


if torch.cuda.is_available():

    print(
        "GPU         :",
        torch.cuda.get_device_name(0),
    )

else:

    print(
        "\nNOTE: GPU is not enabled yet."
    )


print(
    "\nPASS: environment ready."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.3 MB/s eta 0:00:00
ENVIRONMENT
Python      : 3.12.13
PyTorch     : 2.10.0+cu128
NumPy       : 2.0.2
scikit-learn: 1.6.1
anndata     : 0.11.4
CUDA        : True
GPU         : Tesla T4

PASS: environment ready.


Cell 3：检查 E18.5 基础配置并创建 ablation runner

In [3]:
# ============================================================
# Cell 3
# Prepare E18.5 base config and ablation runner
# ============================================================

from pathlib import Path
import shutil
import yaml


BASE_CONFIG = (
    PROJECT_ROOT
    / "configs"
    / "final_clean"
    / "e185_clean_200.yaml"
)


BASE_RUNNER = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp.py"
)


ABLATION_RUNNER = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp_ablation_400.py"
)


assert BASE_CONFIG.exists(), (
    f"Missing config:\n{BASE_CONFIG}"
)

assert BASE_RUNNER.exists(), (
    f"Missing runner:\n{BASE_RUNNER}"
)


# Clean copy of formal runner
shutil.copy2(
    BASE_RUNNER,
    ABLATION_RUNNER,
)


with BASE_CONFIG.open(
    "r",
    encoding="utf-8",
) as f:

    base_cfg = yaml.safe_load(f)


print("=" * 100)
print("E18.5 BASE CONFIG")
print("=" * 100)

print(
    "dataset =",
    base_cfg["experiment"]["dataset"],
)

print(
    "base config =",
    BASE_CONFIG,
)

print(
    "ablation runner =",
    ABLATION_RUNNER,
)

print(
    "\nPASS: clean ablation runner created."
)

E18.5 BASE CONFIG
dataset = E18.5
base config = /kaggle/working/SpaMGCL/SpaMGCL/configs/final_clean/e185_clean_200.yaml
ablation runner = /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_ablation_400.py

PASS: clean ablation runner created.


Cell 4：创建 Fine / Coarse / Linear Fusion 专用 runner

In [4]:
# ============================================================
# Cell 4
# Create dimension-safe MG ablation runner
#
# fine_only:
#     z -> Linear -> g
#
# coarse_only:
#     h -> Linear -> g
#
# linear_fusion:
#     [z,h] -> single Linear -> g
#
# Core src/ is NOT modified.
# ============================================================

from pathlib import Path
import py_compile


MG_RUNNER = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp_mg_ablation_400.py"
)


text = ABLATION_RUNNER.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

import_marker = "import torch\n"

assert import_marker in text


text = text.replace(
    import_marker,
    (
        "import torch\n"
        "from torch import nn\n"
        "import torch.nn.functional as F\n"
    ),
    1,
)


# ------------------------------------------------------------
# Ablation modules
# ------------------------------------------------------------

insert_marker = "def _build_graphs("

assert insert_marker in text


helper_code = r'''
class _FineOnlyFusion(nn.Module):

    def __init__(
        self,
        fine_dim,
        output_dim,
    ):
        super().__init__()

        self.projection = nn.Linear(
            fine_dim,
            output_dim,
        )

    def forward(
        self,
        z,
        h,
    ):
        return F.normalize(
            self.projection(z),
            dim=1,
        )


class _CoarseOnlyFusion(nn.Module):

    def __init__(
        self,
        coarse_dim,
        output_dim,
    ):
        super().__init__()

        self.projection = nn.Linear(
            coarse_dim,
            output_dim,
        )

    def forward(
        self,
        z,
        h,
    ):
        return F.normalize(
            self.projection(h),
            dim=1,
        )


class _LinearFusion(nn.Module):

    def __init__(
        self,
        fine_dim,
        coarse_dim,
        output_dim,
    ):
        super().__init__()

        self.projection = nn.Linear(
            fine_dim + coarse_dim,
            output_dim,
        )

    def forward(
        self,
        z,
        h,
    ):

        x = torch.cat(
            [z, h],
            dim=1,
        )

        return F.normalize(
            self.projection(x),
            dim=1,
        )


def _apply_multigranularity_ablation(
    model,
    config,
):

    section = config.get(
        "multigranularity_ablation",
        {}
    )

    mode = str(
        section.get(
            "mode",
            "full",
        )
    ).lower()


    valid_modes = {
        "full",
        "fine_only",
        "coarse_only",
        "linear_fusion",
    }


    if mode not in valid_modes:

        raise ValueError(
            "Invalid multigranularity "
            f"ablation mode: {mode}"
        )


    if mode == "full":
        return mode


    model_cfg = config.get(
        "model",
        {}
    )


    fine_dim = int(
        model_cfg.get(
            "fine_dim",
            32,
        )
    )

    coarse_dim = int(
        model_cfg.get(
            "coarse_dim",
            32,
        )
    )

    output_dim = int(
        model_cfg.get(
            "representation_dim",
            32,
        )
    )


    device = next(
        model.parameters()
    ).device


    for view_name, encoder in (
        model.view_encoders.items()
    ):

        if mode == "fine_only":

            replacement = _FineOnlyFusion(
                fine_dim,
                output_dim,
            )


        elif mode == "coarse_only":

            replacement = _CoarseOnlyFusion(
                coarse_dim,
                output_dim,
            )


        elif mode == "linear_fusion":

            replacement = _LinearFusion(
                fine_dim,
                coarse_dim,
                output_dim,
            )


        encoder.fusion = (
            replacement.to(device)
        )


    print(
        "MG ablation dimensions:"
    )

    print(
        f"  fine_dim           = {fine_dim}"
    )

    print(
        f"  coarse_dim         = {coarse_dim}"
    )

    print(
        f"  representation_dim = {output_dim}"
    )


    return mode


'''


text = text.replace(
    insert_marker,
    helper_code + insert_marker,
    1,
)


# ------------------------------------------------------------
# Apply after model construction and before optimizer
# ------------------------------------------------------------

old_model_block = '''    model = SpaMGCL(
        input_dims={modality: int(matrix.shape[1]) for modality, matrix in features.items()},
        **model_config,
    ).to(device)
'''


assert old_model_block in text


new_model_block = old_model_block + '''
    mg_ablation_mode = (
        _apply_multigranularity_ablation(
            model,
            config,
        )
    )

    print(
        "Multigranularity ablation mode:",
        mg_ablation_mode,
    )
'''


text = text.replace(
    old_model_block,
    new_model_block,
    1,
)


MG_RUNNER.write_text(
    text,
    encoding="utf-8",
)


py_compile.compile(
    str(MG_RUNNER),
    doraise=True,
)


print("=" * 100)
print("E18.5 MG RUNNER")
print("=" * 100)

print(MG_RUNNER)

print(
    "\nPASS: runner compiled."
)

print(
    "PASS: core src/ unchanged."
)

E18.5 MG RUNNER
/kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_mg_ablation_400.py

PASS: runner compiled.
PASS: core src/ unchanged.


Cell 5：生成 E18.5 的 20 个正式配置

In [5]:
# ============================================================
# Cell 5
# Generate E18.5 ablation configs
#
# 4 variants x 5 seeds = 20 configs
#
# E18.5 FULL loss:
#   lambda_rec     = 1.0
#   lambda_mgcl    = 3.0
#   lambda_cluster = 0.1
#   lambda_spatial = 0.0
# ============================================================

from pathlib import Path
import copy
import yaml


E185_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "e185_ablation_400ep"
)

E185_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


with BASE_CONFIG.open(
    "r",
    encoding="utf-8",
) as f:

    base_cfg = yaml.safe_load(f)


E185_DATASET_NAME = (
    base_cfg["experiment"]["dataset"]
)


VARIANTS = {
    # variant:
    # (MG mode, lambda_mgcl)

    "wo_sample_cl":
        (
            "full",
            0.0,
        ),

    "fine_only":
        (
            "fine_only",
            3.0,
        ),

    "coarse_only":
        (
            "coarse_only",
            3.0,
        ),

    "linear_fusion":
        (
            "linear_fusion",
            3.0,
        ),
}


created = []


for variant, (
    mg_mode,
    lambda_mgcl,
) in VARIANTS.items():

    for seed in range(5):

        cfg = copy.deepcopy(
            base_cfg
        )


        # ----------------------------------------------------
        # Kaggle data root
        # ----------------------------------------------------

        cfg.setdefault(
            "data",
            {}
        )

        cfg["data"]["root"] = str(
            DATA_ROOT
        )


        # ----------------------------------------------------
        # Experiment
        # ----------------------------------------------------

        cfg["experiment"]["seed"] = seed

        cfg["experiment"]["name"] = (
            f"e185_ablation_"
            f"{variant}_"
            f"400ep_seed{seed}"
        )


        # ----------------------------------------------------
        # Frozen protocol
        # ----------------------------------------------------

        cfg["training"]["epochs"] = 400

        cfg["training"][
            "warm_up_epochs"
        ] = 10


        # ----------------------------------------------------
        # E18.5 FULL loss
        # ----------------------------------------------------

        cfg["loss"]["lambda_rec"] = 1.0

        cfg["loss"]["lambda_mgcl"] = (
            lambda_mgcl
        )

        cfg["loss"]["lambda_cluster"] = 0.1

        cfg["loss"]["lambda_spatial"] = 0.0


        # ----------------------------------------------------
        # MG mode
        # ----------------------------------------------------

        cfg[
            "multigranularity_ablation"
        ] = {
            "mode": mg_mode
        }


        # ----------------------------------------------------
        # Frozen readout
        # ----------------------------------------------------

        cfg["clustering"]["embedding"] = (
            "concat_z"
        )

        cfg["clustering"]["n_init"] = 20

        cfg["clustering"][
            "random_state"
        ] = 0


        cfg["refinement"]["enabled"] = True

        cfg["refinement"]["method"] = "bsrr"

        cfg["refinement"][
            "spatial_k"
        ] = 3


        cfg.setdefault(
            "evaluation",
            {}
        )

        cfg["evaluation"][
            "nmi_average_method"
        ] = "max"


        # ----------------------------------------------------
        # Separate E18.5 output root
        # ----------------------------------------------------

        cfg["output"]["root"] = (
            "results_e185_ablation_400"
        )


        output_path = (
            E185_CONFIG_DIR
            / (
                f"e185_ablation_"
                f"{variant}_"
                f"400ep_seed{seed}.yaml"
            )
        )


        with output_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            yaml.safe_dump(
                cfg,
                f,
                sort_keys=False,
            )


        created.append(
            output_path
        )


assert len(created) == 20


print("=" * 100)
print("E18.5 CONFIG GENERATION")
print("=" * 100)

print(
    "Dataset:",
    E185_DATASET_NAME,
)

print(
    "Configs:",
    len(created),
)


for path in created:
    print(path.name)


print(
    "\nPASS: 20 configs generated."
)

E18.5 CONFIG GENERATION
Dataset: E18.5
Configs: 20
e185_ablation_wo_sample_cl_400ep_seed0.yaml
e185_ablation_wo_sample_cl_400ep_seed1.yaml
e185_ablation_wo_sample_cl_400ep_seed2.yaml
e185_ablation_wo_sample_cl_400ep_seed3.yaml
e185_ablation_wo_sample_cl_400ep_seed4.yaml
e185_ablation_fine_only_400ep_seed0.yaml
e185_ablation_fine_only_400ep_seed1.yaml
e185_ablation_fine_only_400ep_seed2.yaml
e185_ablation_fine_only_400ep_seed3.yaml
e185_ablation_fine_only_400ep_seed4.yaml
e185_ablation_coarse_only_400ep_seed0.yaml
e185_ablation_coarse_only_400ep_seed1.yaml
e185_ablation_coarse_only_400ep_seed2.yaml
e185_ablation_coarse_only_400ep_seed3.yaml
e185_ablation_coarse_only_400ep_seed4.yaml
e185_ablation_linear_fusion_400ep_seed0.yaml
e185_ablation_linear_fusion_400ep_seed1.yaml
e185_ablation_linear_fusion_400ep_seed2.yaml
e185_ablation_linear_fusion_400ep_seed3.yaml
e185_ablation_linear_fusion_400ep_seed4.yaml

PASS: 20 configs generated.


Cell 6：严格审计 20 个配置

In [6]:
# ============================================================
# Cell 6
# Audit E18.5 ablation configs
# ============================================================

import yaml


count = 0


for variant, (
    expected_mode,
    expected_mgcl,
) in VARIANTS.items():

    for seed in range(5):

        path = (
            E185_CONFIG_DIR
            / (
                f"e185_ablation_"
                f"{variant}_"
                f"400ep_seed{seed}.yaml"
            )
        )


        with path.open(
            "r",
            encoding="utf-8",
        ) as f:

            cfg = yaml.safe_load(f)


        assert (
            cfg["experiment"]["dataset"]
            == E185_DATASET_NAME
        )

        assert (
            int(
                cfg["experiment"]["seed"]
            )
            == seed
        )

        assert (
            int(
                cfg["training"]["epochs"]
            )
            == 400
        )

        assert (
            int(
                cfg["training"][
                    "warm_up_epochs"
                ]
            )
            == 10
        )


        assert (
            float(
                cfg["loss"]["lambda_rec"]
            )
            == 1.0
        )

        assert (
            float(
                cfg["loss"]["lambda_mgcl"]
            )
            == expected_mgcl
        )

        assert (
            float(
                cfg["loss"][
                    "lambda_cluster"
                ]
            )
            == 0.1
        )

        assert (
            float(
                cfg["loss"][
                    "lambda_spatial"
                ]
            )
            == 0.0
        )


        assert (
            cfg[
                "multigranularity_ablation"
            ]["mode"]
            == expected_mode
        )


        assert (
            cfg["clustering"]["embedding"]
            == "concat_z"
        )

        assert (
            int(
                cfg["clustering"]["n_init"]
            )
            == 20
        )

        assert (
            int(
                cfg["clustering"][
                    "random_state"
                ]
            )
            == 0
        )


        assert (
            cfg["refinement"]["enabled"]
            is True
        )

        assert (
            cfg["refinement"]["method"]
            == "bsrr"
        )

        assert (
            int(
                cfg["refinement"][
                    "spatial_k"
                ]
            )
            == 3
        )


        model_cfg = cfg["model"]

        fine_dim = int(
            model_cfg.get(
                "fine_dim",
                32,
            )
        )

        coarse_dim = int(
            model_cfg.get(
                "coarse_dim",
                32,
            )
        )

        repr_dim = int(
            model_cfg.get(
                "representation_dim",
                32,
            )
        )


        print(
            f"PASS | "
            f"{variant:14s} | "
            f"seed {seed} | "
            f"mgcl={expected_mgcl:.1f} | "
            f"mode={expected_mode:13s} | "
            f"fine={fine_dim} | "
            f"coarse={coarse_dim} | "
            f"g={repr_dim}"
        )


        count += 1


assert count == 20


print(
    "\n" + "=" * 100
)

print(
    "PASS: E18.5 ABLATION PROTOCOL FROZEN"
)

print(
    "20/20 configs verified."
)

print(
    "=" * 100
)

PASS | wo_sample_cl   | seed 0 | mgcl=0.0 | mode=full          | fine=128 | coarse=16 | g=16
PASS | wo_sample_cl   | seed 1 | mgcl=0.0 | mode=full          | fine=128 | coarse=16 | g=16
PASS | wo_sample_cl   | seed 2 | mgcl=0.0 | mode=full          | fine=128 | coarse=16 | g=16
PASS | wo_sample_cl   | seed 3 | mgcl=0.0 | mode=full          | fine=128 | coarse=16 | g=16
PASS | wo_sample_cl   | seed 4 | mgcl=0.0 | mode=full          | fine=128 | coarse=16 | g=16
PASS | fine_only      | seed 0 | mgcl=3.0 | mode=fine_only     | fine=128 | coarse=16 | g=16
PASS | fine_only      | seed 1 | mgcl=3.0 | mode=fine_only     | fine=128 | coarse=16 | g=16
PASS | fine_only      | seed 2 | mgcl=3.0 | mode=fine_only     | fine=128 | coarse=16 | g=16
PASS | fine_only      | seed 3 | mgcl=3.0 | mode=fine_only     | fine=128 | coarse=16 | g=16
PASS | fine_only      | seed 4 | mgcl=3.0 | mode=fine_only     | fine=128 | coarse=16 | g=16
PASS | coarse_only    | seed 0 | mgcl=3.0 | mode=coarse_only   | fine=

Cell 7：E18.5 运行 + 独立指标审计函数

In [7]:
# ============================================================
# Cell 7
# E18.5 run + audit helper
# ============================================================

from pathlib import Path
import subprocess
import yaml
import json
import numpy as np
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


def run_e185_ablation(
    config_path,
):

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA is not available. "
            "Enable Kaggle GPU before training."
        )


    config_path = Path(
        config_path
    )


    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        cfg = yaml.safe_load(f)


    exp_name = (
        cfg["experiment"]["name"]
    )

    seed = int(
        cfg["experiment"]["seed"]
    )

    mode = (
        cfg[
            "multigranularity_ablation"
        ]["mode"]
    )


    run_dir = (
        PROJECT_ROOT
        / cfg["output"]["root"]
        / exp_name
    )


    print(
        "\n" + "=" * 100
    )

    print(
        "RUN:",
        exp_name,
    )

    print(
        "Config:",
        config_path,
    )

    print(
        "Output:",
        run_dir,
    )

    print(
        "=" * 100
    )


    complete = (
        run_dir.exists()
        and (
            run_dir / "metrics.json"
        ).exists()
        and (
            run_dir / "config.yaml"
        ).exists()
        and (
            run_dir / "pred_labels.npy"
        ).exists()
        and (
            run_dir
            / "pred_concat_z_kmeans.npy"
        ).exists()
    )


    if complete:

        print(
            "Existing completed run -> skip."
        )

    else:

        if (
            run_dir.exists()
            and any(run_dir.iterdir())
        ):

            raise RuntimeError(
                "Partial run exists:\n"
                f"{run_dir}"
            )


        subprocess.run(
            [
                "python",
                str(MG_RUNNER),
                "--config",
                str(config_path),
            ],
            cwd=PROJECT_ROOT,
            check=True,
        )


    # --------------------------------------------------------
    # Independent metric recalculation
    # --------------------------------------------------------

    gt = np.load(
        run_dir
        / "gt_labels.npy"
    )

    pred = np.load(
        run_dir
        / "pred_labels.npy"
    )

    pred_raw = np.load(
        run_dir
        / "pred_concat_z_kmeans.npy"
    )


    ari = adjusted_rand_score(
        gt,
        pred,
    )

    nmi = normalized_mutual_info_score(
        gt,
        pred,
        average_method="max",
    )


    raw_ari = adjusted_rand_score(
        gt,
        pred_raw,
    )

    raw_nmi = normalized_mutual_info_score(
        gt,
        pred_raw,
        average_method="max",
    )


    with (
        run_dir
        / "metrics.json"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:

        metrics = json.load(f)


    assert np.isclose(
        ari,
        float(metrics["ARI"]),
        atol=1e-12,
    )

    assert np.isclose(
        nmi,
        float(metrics["NMI"]),
        atol=1e-12,
    )


    print(
        "\n" + "-" * 90
    )

    print(
        f"AUDIT PASS | "
        f"E18.5 | "
        f"{exp_name} | "
        f"seed {seed}"
    )

    print(
        "-" * 90
    )

    print(
        f"Official ARI = {ari:.12f}"
    )

    print(
        f"Official NMI = {nmi:.12f}"
    )

    print(
        f"Raw ARI      = {raw_ari:.12f}"
    )

    print(
        f"Raw NMI      = {raw_nmi:.12f}"
    )

    print(
        f"BSRR ΔARI    = "
        f"{ari - raw_ari:+.12f}"
    )

    print(
        f"BSRR ΔNMI    = "
        f"{nmi - raw_nmi:+.12f}"
    )

    print(
        f"MG mode      = {mode}"
    )


    return {
        "seed": seed,
        "ARI": ari,
        "NMI": nmi,
        "raw_ARI": raw_ari,
        "raw_NMI": raw_nmi,
        "run_dir": run_dir,
    }


print(
    "PASS: E18.5 execution helper ready."
)

PASS: E18.5 execution helper ready.


Cell 8：先跑 4 个 seed0

In [8]:
# ============================================================
# Cell 8
# E18.5 seed0 smoke tests
#
# Run:
#   w/o Sample CL
#   Fine-only
#   Coarse-only
#   Linear fusion
#
# STOP after this cell.
# ============================================================

FULL_SEED0_ARI = 0.437536
FULL_SEED0_NMI = 0.554442


SMOKE_RESULTS = {}


for variant in [
    "wo_sample_cl",
    "fine_only",
    "coarse_only",
    "linear_fusion",
]:

    config_path = (
        E185_CONFIG_DIR
        / (
            f"e185_ablation_"
            f"{variant}_"
            f"400ep_seed0.yaml"
        )
    )


    result = run_e185_ablation(
        config_path
    )


    SMOKE_RESULTS[
        variant
    ] = result


print(
    "\n" + "=" * 100
)

print(
    "E18.5 ABLATION SEED0 SMOKE SUMMARY"
)

print(
    "=" * 100
)


print(
    "\nFull"
)

print(
    f"  ARI = {FULL_SEED0_ARI:.6f}"
)

print(
    f"  NMI = {FULL_SEED0_NMI:.6f}"
)


for variant, result in (
    SMOKE_RESULTS.items()
):

    print(
        f"\n{variant}"
    )

    print(
        f"  ARI  = "
        f"{result['ARI']:.6f}"
    )

    print(
        f"  NMI  = "
        f"{result['NMI']:.6f}"
    )

    print(
        f"  ΔARI = "
        f"{result['ARI'] - FULL_SEED0_ARI:+.6f}"
    )

    print(
        f"  ΔNMI = "
        f"{result['NMI'] - FULL_SEED0_NMI:+.6f}"
    )


print(
    "\n" + "=" * 100
)

print(
    "STOP HERE."
)

print(
    "Do NOT run seeds 1-4 yet."
)

print(
    "=" * 100
)


RUN: e185_ablation_wo_sample_cl_400ep_seed0
Config: /kaggle/working/SpaMGCL/SpaMGCL/configs/e185_ablation_400ep/e185_ablation_wo_sample_cl_400ep_seed0.yaml
Output: /kaggle/working/SpaMGCL/SpaMGCL/results_e185_ablation_400/e185_ablation_wo_sample_cl_400ep_seed0
Multigranularity ablation mode: full
Dataset: E18.5 | spots=2129 | device=cuda
Modalities: RNA, ATAC | label=Combined_Clusters
Spatial graph: shape=(2129, 2129), nnz=7784
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=0.160814 | rec=0.160814 | mgcl=7.668721 | cluster=3.298035 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.967e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.976 | gradC=0.000e+00
epoch 002/400 | total=0.145640 | rec=0.145640 | mgcl=7.668607 | cluster=3.297509 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.733e-03 | neg_count=4530512 | snf_masked_positions=0 | effC

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (14). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)



epoch 372/400 | total=0.084727 | rec=0.001360 | mgcl=7.663406 | cluster=0.833672 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.302e-02 | neg_count=4530512 | snf_masked_positions=0 | effC=13.124 | gradC=7.596e-04
epoch 373/400 | total=0.084769 | rec=0.001411 | mgcl=7.663407 | cluster=0.833585 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.318e-02 | neg_count=4530512 | snf_masked_positions=0 | effC=13.128 | gradC=8.010e-04
epoch 374/400 | total=0.084740 | rec=0.001412 | mgcl=7.663406 | cluster=0.833287 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.340e-02 | neg_count=4530512 | snf_masked_positions=0 | effC=13.124 | gradC=7.685e-04
epoch 375/400 | total=0.084656 | rec=0.001369 | mgcl=7.663406 | cluster=0.832871 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.361e-02 | ne

Cell 9：跑剩余 16 个正式实验

In [9]:
# ============================================================
# Cell 9
# Run remaining E18.5 ablation experiments
#
# Already completed:
#   seed0 for all 4 variants
#
# Now run:
#   seeds 1-4
#
# 4 variants x 4 seeds = 16 new runs
# ============================================================

FORMAL_E185_RESULTS = []


for variant in [
    "wo_sample_cl",
    "fine_only",
    "coarse_only",
    "linear_fusion",
]:

    for seed in range(1, 5):

        config_path = (
            E185_CONFIG_DIR
            / (
                f"e185_ablation_"
                f"{variant}_"
                f"400ep_seed{seed}.yaml"
            )
        )


        result = run_e185_ablation(
            config_path
        )


        FORMAL_E185_RESULTS.append(
            {
                "variant": variant,
                **result,
            }
        )


print(
    "\n" + "=" * 100
)

print(
    "PASS: E18.5 REMAINING ABLATION RUNS COMPLETE"
)

print(
    f"Completed new runs: "
    f"{len(FORMAL_E185_RESULTS)}/16"
)

print(
    "=" * 100
)


RUN: e185_ablation_wo_sample_cl_400ep_seed1
Config: /kaggle/working/SpaMGCL/SpaMGCL/configs/e185_ablation_400ep/e185_ablation_wo_sample_cl_400ep_seed1.yaml
Output: /kaggle/working/SpaMGCL/SpaMGCL/results_e185_ablation_400/e185_ablation_wo_sample_cl_400ep_seed1
Multigranularity ablation mode: full
Dataset: E18.5 | spots=2129 | device=cuda
Modalities: RNA, ATAC | label=Combined_Clusters
Spatial graph: shape=(2129, 2129), nnz=7784
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=0.156004 | rec=0.156004 | mgcl=7.669144 | cluster=3.298389 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.925e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.974 | gradC=0.000e+00
epoch 002/400 | total=0.141978 | rec=0.141978 | mgcl=7.668609 | cluster=3.297887 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.585e-03 | neg_count=4530512 | snf_masked_positions=0 | effC

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (14). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)



epoch 372/400 | total=0.083714 | rec=0.001056 | mgcl=7.663407 | cluster=0.826575 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.295e-02 | neg_count=4530512 | snf_masked_positions=0 | effC=13.034 | gradC=3.138e-04
epoch 373/400 | total=0.083683 | rec=0.001047 | mgcl=7.663406 | cluster=0.826353 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.261e-02 | neg_count=4530512 | snf_masked_positions=0 | effC=13.034 | gradC=3.159e-04
epoch 374/400 | total=0.083654 | rec=0.001041 | mgcl=7.663406 | cluster=0.826129 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.244e-02 | neg_count=4530512 | snf_masked_positions=0 | effC=13.034 | gradC=3.029e-04
epoch 375/400 | total=0.083641 | rec=0.001049 | mgcl=7.663406 | cluster=0.825921 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.219e-02 | ne

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (14). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)



epoch 372/400 | total=0.085455 | rec=0.001410 | mgcl=7.663406 | cluster=0.840454 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.673e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.090 | gradC=5.357e-04
epoch 373/400 | total=0.085410 | rec=0.001375 | mgcl=7.663405 | cluster=0.840346 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.928e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.086 | gradC=6.292e-04
epoch 374/400 | total=0.085372 | rec=0.001354 | mgcl=7.663406 | cluster=0.840173 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.175e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.090 | gradC=6.696e-04
epoch 375/400 | total=0.085386 | rec=0.001379 | mgcl=7.663406 | cluster=0.840065 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.378e-03 | ne

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (14). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)



epoch 372/400 | total=0.082223 | rec=0.001201 | mgcl=7.663406 | cluster=0.810228 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.886e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.392 | gradC=9.808e-04
epoch 373/400 | total=0.082179 | rec=0.001197 | mgcl=7.663408 | cluster=0.809814 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.758e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.372 | gradC=8.318e-04
epoch 374/400 | total=0.082118 | rec=0.001188 | mgcl=7.663407 | cluster=0.809306 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.602e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.386 | gradC=5.583e-04
epoch 375/400 | total=0.082080 | rec=0.001183 | mgcl=7.663406 | cluster=0.808968 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.455e-03 | ne

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (14). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)



epoch 372/400 | total=0.084869 | rec=0.001095 | mgcl=7.663406 | cluster=0.837741 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.629e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.054 | gradC=3.072e-04
epoch 373/400 | total=0.084849 | rec=0.001093 | mgcl=7.663406 | cluster=0.837558 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.015e-02 | neg_count=4530512 | snf_masked_positions=0 | effC=13.056 | gradC=3.484e-04
epoch 374/400 | total=0.084825 | rec=0.001087 | mgcl=7.663406 | cluster=0.837376 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.085e-02 | neg_count=4530512 | snf_masked_positions=0 | effC=13.054 | gradC=3.719e-04
epoch 375/400 | total=0.084800 | rec=0.001079 | mgcl=7.663406 | cluster=0.837209 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.114e-02 | ne

Cell 10：汇总 E18.5 五行正式消融表

In [10]:
# ============================================================
# Cell 10
# E18.5 final ablation summary
#
# Variants:
#   Full
#   w/o Sample CL
#   Fine-only
#   Coarse-only
#   w/o nonlinear fusion
#
# 400 epochs | seeds 0-4 | mean ± std
# std: ddof=0
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


# ============================================================
# Frozen E18.5 Full reference
# seeds 0-4
# ============================================================

FULL_E185 = {
    0: {
        "ARI": 0.437536,
        "NMI": 0.554442,
    },
    1: {
        "ARI": 0.423785,
        "NMI": 0.547109,
    },
    2: {
        "ARI": 0.466861,
        "NMI": 0.572917,
    },
    3: {
        "ARI": 0.331331,
        "NMI": 0.545151,
    },
    4: {
        "ARI": 0.512318,
        "NMI": 0.575264,
    },
}


VARIANT_LABELS = {
    "wo_sample_cl":
        "w/o Sample CL",

    "fine_only":
        "Fine-only",

    "coarse_only":
        "Coarse-only",

    "linear_fusion":
        "w/o nonlinear fusion",
}


rows = []


# ============================================================
# 1. Full
# ============================================================

for seed in range(5):

    rows.append(
        {
            "Dataset":
                "E18.5",

            "Variant":
                "Full",

            "Seed":
                seed,

            "ARI":
                FULL_E185[seed]["ARI"],

            "NMI":
                FULL_E185[seed]["NMI"],

            "Source":
                "frozen_formal_reference",
        }
    )


# ============================================================
# 2. Ablation runs
# ============================================================

for variant_key, variant_label in (
    VARIANT_LABELS.items()
):

    for seed in range(5):

        run_dir = (
            PROJECT_ROOT
            / "results_e185_ablation_400"
            / (
                f"e185_ablation_"
                f"{variant_key}_"
                f"400ep_seed{seed}"
            )
        )


        assert run_dir.exists(), (
            f"Missing run:\n{run_dir}"
        )


        gt_path = (
            run_dir
            / "gt_labels.npy"
        )

        pred_path = (
            run_dir
            / "pred_labels.npy"
        )


        assert gt_path.exists()
        assert pred_path.exists()


        gt = np.load(
            gt_path
        )

        pred = np.load(
            pred_path
        )


        ari = adjusted_rand_score(
            gt,
            pred,
        )

        nmi = (
            normalized_mutual_info_score(
                gt,
                pred,
                average_method="max",
            )
        )


        rows.append(
            {
                "Dataset":
                    "E18.5",

                "Variant":
                    variant_label,

                "Seed":
                    seed,

                "ARI":
                    ari,

                "NMI":
                    nmi,

                "Source":
                    "local_ablation_run",
            }
        )


raw_df = pd.DataFrame(
    rows
)


VARIANT_ORDER = [
    "Full",
    "w/o Sample CL",
    "Fine-only",
    "Coarse-only",
    "w/o nonlinear fusion",
]


raw_df["Variant"] = (
    pd.Categorical(
        raw_df["Variant"],
        categories=VARIANT_ORDER,
        ordered=True,
    )
)


raw_df = (
    raw_df
    .sort_values(
        [
            "Variant",
            "Seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 3. Audit run counts
# ============================================================

counts = (
    raw_df
    .groupby(
        "Variant",
        observed=True,
    )
    .size()
)


print(
    "Run counts:"
)

print(counts)


assert (
    counts == 5
).all(), counts


# ============================================================
# 4. Summary
# ============================================================

summary_rows = []


for variant in VARIANT_ORDER:

    d = (
        raw_df[
            raw_df["Variant"]
            == variant
        ]
        .sort_values("Seed")
    )


    ari = (
        d["ARI"]
        .to_numpy()
    )

    nmi = (
        d["NMI"]
        .to_numpy()
    )


    summary_rows.append(
        {
            "Variant":
                variant,

            "ARI_mean":
                ari.mean(),

            "ARI_std":
                ari.std(
                    ddof=0
                ),

            "NMI_mean":
                nmi.mean(),

            "NMI_std":
                nmi.std(
                    ddof=0
                ),
        }
    )


summary_df = pd.DataFrame(
    summary_rows
)


full_ari = (
    summary_df.loc[
        summary_df["Variant"]
        == "Full",
        "ARI_mean",
    ]
    .iloc[0]
)

full_nmi = (
    summary_df.loc[
        summary_df["Variant"]
        == "Full",
        "NMI_mean",
    ]
    .iloc[0]
)


summary_df[
    "Delta_ARI_vs_Full"
] = (
    summary_df["ARI_mean"]
    - full_ari
)


summary_df[
    "Delta_NMI_vs_Full"
] = (
    summary_df["NMI_mean"]
    - full_nmi
)


summary_df["ARI"] = (
    summary_df.apply(
        lambda r:
            f"{r['ARI_mean']:.6f} "
            f"± "
            f"{r['ARI_std']:.6f}",
        axis=1,
    )
)


summary_df["NMI"] = (
    summary_df.apply(
        lambda r:
            f"{r['NMI_mean']:.6f} "
            f"± "
            f"{r['NMI_std']:.6f}",
        axis=1,
    )
)


# ============================================================
# 5. Display
# ============================================================

print(
    "\n" + "=" * 115
)

print(
    "E18.5 FINAL ABLATION STUDY"
)

print(
    "400 epochs | seeds 0-4 | mean ± std"
)

print(
    "=" * 115
)


print(
    summary_df[
        [
            "Variant",
            "ARI",
            "NMI",
            "Delta_ARI_vs_Full",
            "Delta_NMI_vs_Full",
        ]
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:+.6f}",
    )
)


# ============================================================
# 6. Save
# ============================================================

SUMMARY_DIR = (
    PROJECT_ROOT
    / "ablation_summary_e185_400"
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


raw_path = (
    SUMMARY_DIR
    / "E185_final_ablation_5seeds_RAW.csv"
)

summary_path = (
    SUMMARY_DIR
    / "E185_final_ablation_5seeds_SUMMARY.csv"
)


raw_df.to_csv(
    raw_path,
    index=False,
)


summary_df.to_csv(
    summary_path,
    index=False,
)


print(
    "\nSaved:"
)

print(raw_path)
print(summary_path)


print(
    "\nPASS: E18.5 FINAL "
    "ABLATION SUMMARY VERIFIED."
)

Run counts:
Variant
Full                    5
w/o Sample CL           5
Fine-only               5
Coarse-only             5
w/o nonlinear fusion    5
dtype: int64

E18.5 FINAL ABLATION STUDY
400 epochs | seeds 0-4 | mean ± std
             Variant                 ARI                 NMI  Delta_ARI_vs_Full  Delta_NMI_vs_Full
                Full 0.434366 ± 0.059784 0.558977 ± 0.012745          +0.000000          +0.000000
       w/o Sample CL 0.317722 ± 0.058619 0.465000 ± 0.016373          -0.116644          -0.093977
           Fine-only 0.414456 ± 0.037901 0.563483 ± 0.013756          -0.019910          +0.004507
         Coarse-only 0.359519 ± 0.068484 0.524579 ± 0.013356          -0.074847          -0.034398
w/o nonlinear fusion 0.419517 ± 0.043400 0.564057 ± 0.009468          -0.014850          +0.005080

Saved:
/kaggle/working/SpaMGCL/SpaMGCL/ablation_summary_e185_400/E185_final_ablation_5seeds_RAW.csv
/kaggle/working/SpaMGCL/SpaMGCL/ablation_summary_e185_400/E185_final_ablation_

下一格：生成最终两数据集 Table 3

In [11]:
# ============================================================
# Final combined ablation table
# HLN-A1 + E18.5
#
# 400 epochs | seeds 0-4 | mean ± std
# ============================================================

import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# HLN-A1 final values
# ------------------------------------------------------------

hlna1 = pd.DataFrame([
    {
        "Dataset": "HLN-A1",
        "Variant": "Full",
        "ARI": "0.254705 ± 0.022142",
        "NMI": "0.379401 ± 0.019915",
        "Delta_ARI": 0.000000,
        "Delta_NMI": 0.000000,
    },
    {
        "Dataset": "HLN-A1",
        "Variant": "w/o Sample CL",
        "ARI": "0.195046 ± 0.020640",
        "NMI": "0.308321 ± 0.021377",
        "Delta_ARI": -0.059659,
        "Delta_NMI": -0.071080,
    },
    {
        "Dataset": "HLN-A1",
        "Variant": "Fine-only",
        "ARI": "0.236880 ± 0.024257",
        "NMI": "0.351806 ± 0.009865",
        "Delta_ARI": -0.017825,
        "Delta_NMI": -0.027594,
    },
    {
        "Dataset": "HLN-A1",
        "Variant": "Coarse-only",
        "ARI": "0.236784 ± 0.012546",
        "NMI": "0.360722 ± 0.015366",
        "Delta_ARI": -0.017921,
        "Delta_NMI": -0.018679,
    },
    {
        "Dataset": "HLN-A1",
        "Variant": "w/o nonlinear fusion",
        "ARI": "0.243713 ± 0.024313",
        "NMI": "0.364073 ± 0.018845",
        "Delta_ARI": -0.010992,
        "Delta_NMI": -0.015328,
    },
])


# ------------------------------------------------------------
# E18.5 final values
# ------------------------------------------------------------

e185 = pd.DataFrame([
    {
        "Dataset": "E18.5",
        "Variant": "Full",
        "ARI": "0.434366 ± 0.059784",
        "NMI": "0.558977 ± 0.012745",
        "Delta_ARI": 0.000000,
        "Delta_NMI": 0.000000,
    },
    {
        "Dataset": "E18.5",
        "Variant": "w/o Sample CL",
        "ARI": "0.317722 ± 0.058619",
        "NMI": "0.465000 ± 0.016373",
        "Delta_ARI": -0.116644,
        "Delta_NMI": -0.093977,
    },
    {
        "Dataset": "E18.5",
        "Variant": "Fine-only",
        "ARI": "0.414456 ± 0.037901",
        "NMI": "0.563483 ± 0.013756",
        "Delta_ARI": -0.019910,
        "Delta_NMI": +0.004507,
    },
    {
        "Dataset": "E18.5",
        "Variant": "Coarse-only",
        "ARI": "0.359519 ± 0.068484",
        "NMI": "0.524579 ± 0.013356",
        "Delta_ARI": -0.074847,
        "Delta_NMI": -0.034398,
    },
    {
        "Dataset": "E18.5",
        "Variant": "w/o nonlinear fusion",
        "ARI": "0.419517 ± 0.043400",
        "NMI": "0.564057 ± 0.009468",
        "Delta_ARI": -0.014850,
        "Delta_NMI": +0.005080,
    },
])


combined = pd.concat(
    [hlna1, e185],
    ignore_index=True,
)


print("=" * 125)
print("FINAL TWO-DATASET ABLATION TABLE")
print("400 epochs | seeds 0-4 | mean ± std")
print("=" * 125)

print(
    combined.to_string(
        index=False,
        float_format=lambda x: f"{x:+.6f}",
    )
)


OUT_DIR = (
    PROJECT_ROOT
    / "final_ablation_tables"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


out_path = (
    OUT_DIR
    / "Table3_HLNA1_E185_ablation_FINAL.csv"
)

combined.to_csv(
    out_path,
    index=False,
)


print("\nSaved:")
print(out_path)

print(
    "\nPASS: two-dataset ablation table finalized."
)

FINAL TWO-DATASET ABLATION TABLE
400 epochs | seeds 0-4 | mean ± std
Dataset              Variant                 ARI                 NMI  Delta_ARI  Delta_NMI
 HLN-A1                 Full 0.254705 ± 0.022142 0.379401 ± 0.019915  +0.000000  +0.000000
 HLN-A1        w/o Sample CL 0.195046 ± 0.020640 0.308321 ± 0.021377  -0.059659  -0.071080
 HLN-A1            Fine-only 0.236880 ± 0.024257 0.351806 ± 0.009865  -0.017825  -0.027594
 HLN-A1          Coarse-only 0.236784 ± 0.012546 0.360722 ± 0.015366  -0.017921  -0.018679
 HLN-A1 w/o nonlinear fusion 0.243713 ± 0.024313 0.364073 ± 0.018845  -0.010992  -0.015328
  E18.5                 Full 0.434366 ± 0.059784 0.558977 ± 0.012745  +0.000000  +0.000000
  E18.5        w/o Sample CL 0.317722 ± 0.058619 0.465000 ± 0.016373  -0.116644  -0.093977
  E18.5            Fine-only 0.414456 ± 0.037901 0.563483 ± 0.013756  -0.019910  +0.004507
  E18.5          Coarse-only 0.359519 ± 0.068484 0.524579 ± 0.013356  -0.074847  -0.034398
  E18.5 w/o nonlinear

In [12]:
# ============================================================
# FINAL CELL
# Archive E18.5 ablation before closing Kaggle
# ============================================================

from pathlib import Path
import shutil
import hashlib
import pandas as pd


ARCHIVE_ROOT = (
    PROJECT_ROOT
    / "SpaMGCL_E185_Ablation_400ep_5seeds_FINAL"
)


if ARCHIVE_ROOT.exists():
    shutil.rmtree(
        ARCHIVE_ROOT
    )


ARCHIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. Summary tables
# ============================================================

summary_src = (
    PROJECT_ROOT
    / "ablation_summary_e185_400"
)

if summary_src.exists():

    shutil.copytree(
        summary_src,
        ARCHIVE_ROOT
        / "ablation_summary_e185_400",
    )


# ============================================================
# 2. Combined final table, if already created
# ============================================================

combined_src = (
    PROJECT_ROOT
    / "final_ablation_tables"
)

if combined_src.exists():

    shutil.copytree(
        combined_src,
        ARCHIVE_ROOT
        / "final_ablation_tables",
    )


# ============================================================
# 3. Formal configs
# ============================================================

config_src = (
    PROJECT_ROOT
    / "configs"
    / "e185_ablation_400ep"
)

assert config_src.exists(), (
    f"Missing config directory:\n{config_src}"
)


shutil.copytree(
    config_src,
    ARCHIVE_ROOT
    / "configs"
    / "e185_ablation_400ep",
)


# ============================================================
# 4. Exact MG ablation runner used
# ============================================================

runner_src = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp_mg_ablation_400.py"
)

assert runner_src.exists()


runner_dst = (
    ARCHIVE_ROOT
    / "experiments"
)

runner_dst.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copy2(
    runner_src,
    runner_dst
    / runner_src.name,
)


# ============================================================
# 5. Essential evidence from all 20 E18.5 runs
#
# Do NOT copy large checkpoints / embeddings.
# ============================================================

runs_src = (
    PROJECT_ROOT
    / "results_e185_ablation_400"
)

assert runs_src.exists()


runs_dst = (
    ARCHIVE_ROOT
    / "results_e185_ablation_400"
)

runs_dst.mkdir(
    parents=True,
    exist_ok=True,
)


ESSENTIAL_FILES = [
    "metrics.json",
    "config.yaml",
    "manifest.json",
    "gt_labels.npy",
    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
]


run_count = 0


for run_dir in sorted(
    runs_src.iterdir()
):

    if not run_dir.is_dir():
        continue


    # Only E18.5 formal ablation runs
    if not run_dir.name.startswith(
        "e185_ablation_"
    ):
        continue


    dst = (
        runs_dst
        / run_dir.name
    )

    dst.mkdir(
        parents=True,
        exist_ok=True,
    )


    for filename in ESSENTIAL_FILES:

        src_file = (
            run_dir
            / filename
        )

        if src_file.exists():

            shutil.copy2(
                src_file,
                dst / filename,
            )


    run_count += 1


print(
    "Archived run directories:",
    run_count,
)


assert run_count == 20, (
    f"Expected 20 runs, found {run_count}"
)


# ============================================================
# 6. SHA256 manifest
# ============================================================

hash_rows = []


for path in sorted(
    ARCHIVE_ROOT.rglob("*")
):

    if not path.is_file():
        continue


    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b"",
        ):

            h.update(block)


    hash_rows.append(
        {
            "file":
                str(
                    path.relative_to(
                        ARCHIVE_ROOT
                    )
                ),

            "sha256":
                h.hexdigest(),
        }
    )


pd.DataFrame(
    hash_rows
).to_csv(
    ARCHIVE_ROOT
    / "SHA256_MANIFEST.csv",
    index=False,
)


# ============================================================
# 7. ZIP
# ============================================================

zip_path = shutil.make_archive(
    str(ARCHIVE_ROOT),
    "zip",
    root_dir=ARCHIVE_ROOT,
)


print(
    "\n" + "=" * 100
)

print(
    "E18.5 ABLATION ARCHIVE COMPLETE"
)

print(
    "=" * 100
)

print(
    "ZIP:"
)

print(
    zip_path
)

print(
    "\nPASS: 20/20 E18.5 runs archived."
)

print(
    "You can download this ZIP "
    "and close the Kaggle session."
)

Archived run directories: 20

E18.5 ABLATION ARCHIVE COMPLETE
ZIP:
/kaggle/working/SpaMGCL/SpaMGCL/SpaMGCL_E185_Ablation_400ep_5seeds_FINAL.zip

PASS: 20/20 E18.5 runs archived.
You can download this ZIP and close the Kaggle session.
